[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/06_feeding_the_beast.ipynb)

# ⚒️ Act II · Quest 06 — Feeding the Beast

*Models are only as good as their food: Datasets, DataLoaders, splits, and augmentation.*

⬅️ [05_your_first_real_network](05_your_first_real_network.ipynb)  •  [07_eyes_convolutions](07_eyes_convolutions.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## Why not just `model(all_the_data)`?

Three reasons real training feeds **mini-batches**:
1. **Memory** — datasets don't fit in RAM/VRAM.
2. **Speed** — more parameter updates per pass over the data.
3. **Noise is good** — slightly noisy gradients help escape bad valleys.

PyTorch splits the job cleanly:
- **`Dataset`** — "how do I fetch sample *i*?" (`__len__` + `__getitem__`)
- **`DataLoader`** — batching, shuffling, parallel workers, on top of any Dataset.

### Meet the course mascots: the Glyphs ✕ ◯ ┼ ╱

In [ ]:
# --- The Glyph dataset: ✕ ◯ ┼ ╱  (self-contained, no torchvision needed) ----
GLYPHS = ["cross", "ring", "plus", "slash"]

def _draw_glyph(cls, size=20, rng=None):
    rng = rng or torch.Generator().manual_seed(0)
    img = torch.zeros(size, size)
    ys, xs = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    jx = int(torch.randint(-2, 3, (1,), generator=rng))   # random jitter
    jy = int(torch.randint(-2, 3, (1,), generator=rng))
    c = size // 2
    if cls == 0:    # cross ✕ : two diagonals
        img[((xs - ys).abs() + (jx - jy)).abs() <= 1] = 1.0
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    elif cls == 1:  # ring ◯
        r2 = (xs - c - jx) ** 2 + (ys - c - jy) ** 2
        img[(r2 >= 25) & (r2 <= 49)] = 1.0
    elif cls == 2:  # plus ┼
        img[(ys - c - jy).abs() <= 1] = 1.0
        img[(xs - c - jx).abs() <= 1] = 1.0
    else:           # slash ╱ : one diagonal
        img[((xs + ys - size + 1) + jx + jy).abs() <= 1] = 1.0
    img = img + 0.08 * torch.randn(size, size, generator=rng)
    return img.clamp(0, 1)

def make_glyphs(n_per_class=300, size=20, seed=0):
    rng = torch.Generator().manual_seed(seed)
    X = torch.stack([_draw_glyph(c, size, rng) for c in range(4) for _ in range(n_per_class)])
    y = torch.arange(4).repeat_interleave(n_per_class)
    perm = torch.randperm(len(y), generator=rng)
    return X[perm].unsqueeze(1), y[perm]   # (N, 1, 20, 20), (N,)

In [ ]:
X, y = make_glyphs(n_per_class=300)
print("images:", X.shape, "| labels:", y.shape, "| classes:", GLYPHS)

fig, ax = plt.subplots(2, 8, figsize=(12, 3.2))
for i in range(16):
    ax[i // 8, i % 8].imshow(X[i, 0], cmap="gray")
    ax[i // 8, i % 8].set_title(GLYPHS[y[i]], fontsize=8)
    ax[i // 8, i % 8].axis("off")
plt.suptitle("The Glyph dataset — jittered, noisy, self-contained"); plt.show()

### Wrap it in a `Dataset`, feed it through a `DataLoader`

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

ds = TensorDataset(X, y)              # quickest wrapper for in-memory tensors
print("len:", len(ds), "| sample 0 shapes:", ds[0][0].shape, ds[0][1])

loader = DataLoader(ds, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print("one batch:", xb.shape, yb.shape, "| batches per epoch:", len(loader))

### The sacred split

You **must** hold out data the model never trains on — otherwise you're grading a student on
questions they memorized. `random_split` does it in one line.

In [ ]:
train_ds, val_ds, test_ds = random_split(ds, [900, 150, 150],
                                         generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)
print(f"train {len(train_ds)} | val {len(val_ds)} | test {len(test_ds)}")
print("train: learn | val: tune choices | test: touch ONCE at the very end")

### Writing your own `Dataset` — the pattern you'll reuse forever

In [ ]:
class GlyphDataset(Dataset):
    """A Dataset that generates glyphs on the fly, with optional augmentation."""
    def __init__(self, n=1200, augment=False, seed=0):
        self.X, self.y = make_glyphs(n // 4, seed=seed)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        img, label = self.X[i], self.y[i]
        if self.augment:
            if random.random() < 0.5:
                img = img.flip(-1)                          # mirror
            img = img.roll(random.randint(-2, 2), dims=-1)   # small shift
            img = (img + 0.05 * torch.randn_like(img)).clamp(0, 1)
        return img, label

aug = GlyphDataset(augment=True)
fig, ax = plt.subplots(1, 6, figsize=(9, 1.8))
for i in range(6):
    img, lbl = aug[0]           # same index -> different augmented views!
    ax[i].imshow(img[0], cmap="gray"); ax[i].axis("off")
plt.suptitle("Augmentation: one sample, many views (free extra data)"); plt.show()

### Does batch size matter? See for yourself

In [ ]:
import time

def one_epoch(batch_size):
    model = nn.Sequential(nn.Flatten(), nn.Linear(400, 64), nn.ReLU(), nn.Linear(64, 4))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    t0 = time.time()
    for xb, yb in loader:
        opt.zero_grad()
        F.cross_entropy(model(xb), yb).backward()
        opt.step()
    return (time.time() - t0) * 1000, len(loader)

for bs in [4, 32, 256]:
    ms, nb = one_epoch(bs)
    print(f"batch_size {bs:4d}: {nb:4d} updates/epoch, {ms:7.1f} ms/epoch")
print("\nsmall batches = many noisy updates; big batches = few smooth ones. 32–256 is the usual sweet spot.")

> 🧵 **`num_workers`**: on Linux/Colab, `DataLoader(..., num_workers=2)` loads batches in
> parallel processes. On Windows notebooks keep it `0` (multiprocessing + notebooks don't mix well).

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda n: n == 19,
    "a Dataset with 1234 samples and batch_size=64, drop_last=False -> how many batches? ceil(1234/64)")
_register("own_dataset", 20,
    lambda d: (lambda s: hasattr(d, "__len__") and len(d) == 100
               and torch.is_tensor(s[0]) and s[0].shape == (3,)
               and torch.is_tensor(s[1]) and s[1].item() == int(s[0].sum().item() > 0))(d[0]),
    "Dataset of 100 samples: x = a (3,) float tensor, y = tensor(1) if x.sum() > 0 else tensor(0)")
_register("split_sizes", 10,
    lambda sizes: list(sizes) == [800, 100, 100],
    "an 80/10/10 split of 1000 samples — submit [train, val, test] sizes")
_register("aug_effect", 10,
    lambda s: s.strip().lower() in ("overfitting", "overfit"),
    "one word: augmentation mainly fights ______")

In [ ]:
check("warmup", 19)

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `own_dataset` (20 XP) — write a `Dataset` of 100 samples where `x` is a random `(3,)` tensor and `y = 1 if x.sum() > 0 else 0` (as a tensor); submit an instance.
- `split_sizes` (10 XP) — split sizes for 80/10/10 of 1000 samples.
- `aug_effect` (10 XP) — one word: what does augmentation mainly fight?

In [ ]:
# ⚔️ your attempts here...

# xp_report()